# Notebook 05b: Importing External Job Postings (BA Job Search API)

In this notebook, German-language job postings are automatically collected via the Job Search API of the Federal Employment Agency (https://jobsuche.api.bund.dev/, 12/20/2025), as the availability of German datasets is very limited. The ads are processed as an external text source in Notebook 05 and used for subsequent skill extraction and profile expansion based on the KldB.

Output:
- Raw dataset of all collected ads
- Filtered dataset with usable full texts

## 1. Setup

In [1]:
# Imports + Paths
from pathlib import Path
import requests
import pandas as pd
import time
import json
import re
from bs4 import BeautifulSoup

# Project Root
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)

# Destination folder for external job postings
DATA_RAW_EXTERNAL = PROJECT_ROOT / "data" / "raw_external"
JOB_ADS_DIR = DATA_RAW_EXTERNAL / "job_ads"
JOB_ADS_DIR.mkdir(parents=True, exist_ok=True)

print("Job Ads Output Dir:", JOB_ADS_DIR)

PROJECT_ROOT: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
Job Ads Output Dir: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\job_ads


## 2. Data Source: BA (Federal Employment Agency) Job Search API

The Federal Employment Agency's Job Search API is used as the primary source for German-language job postings.
The API provides structured metadata for job postings as well as an external URL to the original posting, from which the full text can be extracted.

In [2]:
# API configuration
BASE = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4"

HEADERS = {"accept": "application/json", "X-API-Key": "jobboerse-jobsuche" # Öffentliche BA Jobsuche API, Key ist für diesen Endpunkt erforderlich
           }

## 3. Search Strategy

Since a Germany-wide, non-targeted search is limited in scope, we use thematic search terms (“seeds”). These cover a variety of professions and fields of activity to achieve the broadest possible range of content. The results are not representative of the overall market (sampling bias due to search terms).

In [3]:
# expanded, logically structured
SEEDS = [
    # Health and Social Services
    "Pflege", "Gesundheits", "Krankenpfleger", "Altenpfleger", "Therapie", "Physio", "Ergotherapie", "Erzieher", "Sozialpädagogik", "Kita",

    # Retail/Services
    "Verkauf", "Einzelhandel", "Markt", "Gastronomie", "Küche", "Koch", "Service", "Hotel", 
    
    # Office/Administration/Finance
    "Büro", "Sachbearbeitung", "Verwaltung", "Assistenz", "Sekretariat", "Buchhaltung", "Finanz", "Controlling", "Kaufmann", "Industriekaufmann", "Personal", "HR",

    # Logistics/Transport
    "Logistik", "Lager", "Kommissionierung", "Stapler", "Versand", "Spedition", "Fahrer", "Zusteller",

    # Crafts/Construction/Technology
    "Bau", "Maurer", "Maler", "Dachdecker", "Tischler", "Schreiner", "Metall", "Schlosser", "Elektriker", "Elektro", "Mechatroniker", "Mechaniker", "Anlagen", "Sanitär", "Heizung", "Klima",

    # Industry/Manufacturing
    "Produktion", "Fertigung", "Montage", "Maschinenbediener", "Qualität", "Schicht",

    # Agriculture/Environment
    "Landwirtschaft", "Garten", "Gärtner", "Forst", "Tierpflege",

    # Security/Public Sector
    "Sicherheit", "Hausmeister", "Öffentlicher Dienst", "Militär", "Soldat",
]

## 4. Utility Functions

- Retrieving job postings via the API
- Extracting full text from external job sites

In [4]:
# Find jobs
def fetch_jobs(page: int, size: int, was: str):
    params = {
        "page": page,
        "size": size,
        "was": was
    }
    r = requests.get(f"{BASE}/jobs", headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("stellenangebote", [])

In [5]:
# Extract full texts
def extract_fulltext_from_job_page(url: str, timeout=(5, 20), max_retries=2): # timeout=(connect_timeout, read_timeout): connect_timeout: 5s; read_timeout: 20s
    headers = {"User-Agent": "Mozilla/5.0"}

    for attempt in range(max_retries + 1):
        try:
            resp = requests.get(url, timeout=timeout, headers=headers)
            if resp.status_code != 200:
                return None, f"http_{resp.status_code}"

            soup = BeautifulSoup(resp.text, "html.parser")

            # JSON-LD is preferred for full-text extraction
            for script in soup.find_all("script", type="application/ld+json"):
                try:
                    data = json.loads(script.get_text(strip=True))
                except Exception:
                    continue

                objs = data if isinstance(data, list) else [data]
                for obj in objs:
                    if isinstance(obj, dict) and obj.get("@type") == "JobPosting":
                        desc = obj.get("description")
                        if desc:
                            text = BeautifulSoup(desc, "html.parser").get_text(" ", strip=True)
                            text = re.sub(r"\s+", " ", text).strip()
                            return text, "jsonld"

            # Fallback: visible text
            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()
            text = soup.get_text(" ", strip=True)
            text = re.sub(r"\s+", " ", text).strip()

            if len(text) < 300:
                return None, "fallback_too_short"
            return text, "fallback_visible_text" # Fallback uses visible page text

        # Use Try/Except, otherwise the program will terminate when accessing certain external pages
        except requests.exceptions.ReadTimeout:
            if attempt < max_retries: # Page is frozen -> retry, otherwise skip
                time.sleep(1.5 * (attempt + 1))
                continue
            return None, "timeout_read"

        except requests.exceptions.ConnectTimeout:
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue
            return None, "timeout_connect"

        except requests.exceptions.RequestException:
            return None, "request_exception" # DNS, SSL, ConnectionError, etc.

        except Exception:
            return None, "unknown_exception"

## 5.  Data Collection

The following parameters:
- Goal: approx. 3,000–5,000 usable full-text documents
- Multiple seeds
- Moderate number of pages per seed
- Deduplicated by reference number/URL

In [6]:
TARGET_FULLTEXT = 4000  # Target number of usable full-text documents
PAGES_PER_SEED = 5
SIZE = 50
SLEEP = 0.15
MAX_FULLTEXT_PER_SEED = 100  # Maximum number of usable full-text documents per seed (for balanced distribution)

max_retries = 1
timeout = (4, 12)

rows = []
seen = set()
fulltext_count = 0

# Counts usable full texts per seed
seed_fulltext = {s: 0 for s in SEEDS}

for seed in SEEDS:
    # If the seed limit has already been reached, skip
    if seed_fulltext[seed] >= MAX_FULLTEXT_PER_SEED:
        continue

    print(f"\n Seed gestartet: {seed} ")

    for page in range(1, PAGES_PER_SEED + 1):
        # If the seed limit is reached, do not fetch any more pages
        if seed_fulltext[seed] >= MAX_FULLTEXT_PER_SEED:
            break

        jobs = fetch_jobs(page=page, size=SIZE, was=seed)
        # Page Info
        print(f"Seed='{seed}' Page={page} Jobs={len(jobs)} | SeedGood={seed_fulltext[seed]} | TotalGood={fulltext_count}")

        for j in jobs:
            # Target
            if fulltext_count >= TARGET_FULLTEXT:
                break

            # Seed Limit
            if seed_fulltext[seed] >= MAX_FULLTEXT_PER_SEED:
                break

            refnr = j.get("refnr")
            url = j.get("externeUrl")

            key = refnr or url # Prefer "dedupe refnr", otherwise use "externalUrl"; prevents duplicate entries across multiple seeds/pages
            if not key or key in seen:
                continue
            seen.add(key)

            if not url:
                continue

            text, method = extract_fulltext_from_job_page(url)

            rows.append({
                "refnr": refnr,
                "jobtitel": j.get("titel") or j.get("beruf"),
                "beruf": j.get("beruf"),
                "volltext": text,
                "volltext_methode": method,
                "seed": seed,
                "externeUrl": url # No company information
            })

            # counts usable full-text documents
            if text and len(text) > 300: # Minimum length, excluding short sides
                fulltext_count += 1
                seed_fulltext[seed] += 1

            time.sleep(SLEEP)

        if fulltext_count >= TARGET_FULLTEXT:
            break

    # End pages-loop
    if fulltext_count >= TARGET_FULLTEXT:
        break

print("\nFertig")
print("Gesamt brauchbare Volltexte:", fulltext_count)
print("Gesamt gesammelte Zeilen (raw):", len(rows))

# Overview of how full the seeds are
filled = sum(1 for s in SEEDS if seed_fulltext[s] > 0)
print("Seeds mit mind. 1 Volltext:", filled, "von", len(SEEDS))


 Seed gestartet: Pflege 
Seed='Pflege' Page=1 Jobs=50 | SeedGood=0 | TotalGood=0
Seed='Pflege' Page=2 Jobs=50 | SeedGood=13 | TotalGood=13
Seed='Pflege' Page=3 Jobs=50 | SeedGood=26 | TotalGood=26
Seed='Pflege' Page=4 Jobs=50 | SeedGood=31 | TotalGood=31
Seed='Pflege' Page=5 Jobs=50 | SeedGood=39 | TotalGood=39

 Seed gestartet: Gesundheits 
Seed='Gesundheits' Page=1 Jobs=50 | SeedGood=0 | TotalGood=52
Seed='Gesundheits' Page=2 Jobs=50 | SeedGood=7 | TotalGood=59
Seed='Gesundheits' Page=3 Jobs=50 | SeedGood=54 | TotalGood=106
Seed='Gesundheits' Page=4 Jobs=50 | SeedGood=63 | TotalGood=115
Seed='Gesundheits' Page=5 Jobs=50 | SeedGood=75 | TotalGood=127

 Seed gestartet: Krankenpfleger 
Seed='Krankenpfleger' Page=1 Jobs=50 | SeedGood=0 | TotalGood=152
Seed='Krankenpfleger' Page=2 Jobs=50 | SeedGood=2 | TotalGood=154
Seed='Krankenpfleger' Page=3 Jobs=50 | SeedGood=5 | TotalGood=157
Seed='Krankenpfleger' Page=4 Jobs=50 | SeedGood=20 | TotalGood=172
Seed='Krankenpfleger' Page=5 Jobs=50 | S

## 6. Saving the Results

Two files are generated:
- Raw dataset (all records)
- Filtered dataset with usable full texts

In [7]:
df = pd.DataFrame(rows)

raw_path = JOB_ADS_DIR / "ba_job_ads_raw.csv"
full_path = JOB_ADS_DIR / "ba_job_ads_fulltext.csv"

df.to_csv(raw_path, index=False, encoding="utf-8-sig")

df_full = df[df["volltext"].notna() & (df["volltext"].str.len() > 300)]
df_full.to_csv(full_path, index=False, encoding="utf-8-sig")

print("Gesamtanzeigen:", len(df))
print("Volltexte:", len(df_full))
print("Gespeichert in:", JOB_ADS_DIR)

Gesamtanzeigen: 5414
Volltexte: 2959
Gespeichert in: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\job_ads


## 7. Conclusion: Notebook 05 b

- This notebook generated a reproducible, German-language dataset of external job postings.
- The data will be used later for skill extraction, as well as for supplementing and expanding existing KldB profiles.
- The ads are processed as an external text source in Notebook 05 and used for subsequent skill extraction and profile expansion based on the KldB.
- A total of 5,902 ads were collected during the run, 3,505 of which contained usable full text.